In [ ]:
import numpy as np
from phisolve.refiners.pdqp import PDQP
from phisolve.problems import BoxQP, LCQP
import jax, scipy
from jax import numpy as jnp

In [ ]:
seed = 42

np.random.seed(seed)

n = 50
U = scipy.stats.ortho_group(n, seed).rvs()
eig = np.random.normal(-10, 5, (n))
Q = U.T @ np.diag(eig) @ U
w = np.random.normal(0, 10, (n))
lbs = np.random.uniform(-10, 10, (n))
gaps = np.random.uniform(0.1, 10, (n))
ubs = lbs + gaps
boxqp = BoxQP(Q, w, bounds=(lbs, ubs))

In [ ]:
n_shots = 100
n_steps = 10000
seed = 42
device = "cpu"

In [ ]:
import jax
from phisolve.utils.jax_utils import jax_device
jax.config.update("jax_platforms", jax_device(device))

In [ ]:
lcqp = LCQP(Q, w, bounds=(lbs, ubs))
samples = np.random.normal(0, 1, (n_shots, lcqp.nvar))
pdqp = PDQP(problem=lcqp, device=device, iterations=500, max_K=500)
res = pdqp.pdqp_main(samples)
objs = jax.vmap(lcqp.obj)(res[0])
maxvios = jax.vmap(lambda x: jnp.max(jnp.concat((lbs - x, x - ubs, jnp.zeros(1)))))(res[0])
feas = maxvios < 1e-4
minima = np.min(objs[feas])
minimizer = np.argmin(objs[feas])
succ = objs[feas] <= minima + 1e-4
minimizer_vios = maxvios[feas][minimizer]
succ_prob = np.sum(succ) / n_shots
print(minima, minimizer_vios, succ_prob)
assert minima < -21338

In [ ]:
from phisolve.solvers.phi_miqp import PhiMIQPParams, PhiMIQP
from phisolve.backends.commons import BackendParams
from phisolve.refiners.jax_adam import JaxAdam

backend_params = BackendParams(n_shots=n_shots, n_steps=n_steps, seed=seed, device=device, slow_a=False)
refiner = JaxAdam(device=device).refine
solver_params = PhiMIQPParams(refine=refiner, backend_params=backend_params)

solver = PhiMIQP(boxqp)
res = solver.run(solver_params)

xs, cnts = res.refined_samples, res.sample_counts

objs = jax.vmap(boxqp.obj)(xs)
maxvios = jax.vmap(lambda x: jnp.max(jnp.concat((lbs - x, x - ubs, jnp.zeros(1)))))(xs)
feas = maxvios < 1e-4
minima = np.min(objs[feas])
minimizer = np.argmin(objs[feas])
minimizer_vios = maxvios[feas][minimizer]
print(minima, minimizer_vios, res.succ_prob())
assert minima < -21736

In [ ]:
inbox = np.sum(cnts[feas])
assert inbox == n_shots